# 09 | Simulador da aplicação — somente HTTP

POC semissintética • contrato provisório • sem homologação acadêmica.
Execute em kernel reiniciado, de cima para baixo. As credenciais vêm do ambiente.


**Objetivo:** consumir a proposta persistida sem SQL, treino ou SDK de storage. **Entradas:** contexto salvo em08; override explícito opcional. **Esperado:** paginação completa, hashes e schema válidos.

In [1]:
import os, json
import pandas as pd
from IPython.display import display, Markdown
from src.workbench.application_client import call, context, save_context

from src.workbench.application_client import select_proposal,fetch,table,explorer
PROPOSAL_ID=None  # Opcional: UUID escolhido na lista abaixo; contexto08 é automático.
display(pd.DataFrame(call('GET','/v2/proposals')))
proposal_id=select_proposal(PROPOSAL_ID)
payload=fetch(proposal_id,page_size=13)
display({'proposal_id':proposal_id,'ofertas':len(payload['offers']),'origens':sum(len(o['activities']) for o in payload['offers']),
         'schema':payload['schema_version'],'hash_proposta':payload['proposal_sha256'],'origem':'API HTTP → PostgreSQL'})
display(table(payload['offers']))


,proposal_id,cycle_code,revision,status,generated_at,snapshot_id
0,41f15eac-1dc7-44e0-b360-db0e50da8406,2024-2,1,DRAFT_PENDING_COORDINATOR,2026-11-05T06:29:11.360007+00:00,9158d4db-c80b-5638-8aa0-9467c60f6107
1,15290b2a-8ee6-4a99-996e-320a9c2a5888,2024-2,1,DRAFT_PENDING_COORDINATOR,2026-11-05T06:21:59.977553+00:00,9158d4db-c80b-5638-8aa0-9467c60f6107
2,a249d49e-a8f6-474c-9e7e-fbbec7d4d0e1,2024-2,1,DRAFT_PENDING_COORDINATOR,2026-11-05T06:07:15.460311+00:00,9158d4db-c80b-5638-8aa0-9467c60f6107
3,ac8b4d93-aaa5-4d94-85de-557aa955c8ca,2024-2,1,DRAFT_PENDING_COORDINATOR,2026-11-05T05:55:54.389466+00:00,9158d4db-c80b-5638-8aa0-9467c60f6107


{'proposal_id': '41f15eac-1dc7-44e0-b360-db0e50da8406',
 'ofertas': 70,
 'origens': 107,
 'schema': 'poc-planilha-turmas-v2',
 'hash_proposta': 'a98ce02339dff3fa7c42bbc2684d6024e0874530ce168f3e95c8107fe48eb9f9',
 'origem': 'API HTTP → PostgreSQL'}

,offer_id,Estabelecimento,Escola,Curso,Grade,Ingresso,Coorte,Período,Turno,Disciplina,Descrição,Regular,Dependência,Eletiva,Total_previsto,Atividades,Unificação,Status
0,014f59b5-b159-5130-a643-d8e0194c7bd1,Curitiba,Politécnica,2248,CECP 2021/1,None,None,8,Manhã,PCP108A,Inteligência Artificial e Computacional,NaN,NaN,NaN,NaN,1,NOT_EVALUATED,QUARANTINED
1,01900fe7-e2cb-5f66-bab3-7a168d7a7f2a,Curitiba,Politécnica,2348,CECP 2021/1,2021-1,5c55b155-af77-5e52-b08c-e152afc1dde4,8,Noite,PCP106A,Estágio Curricular de Engenharia de Computação,34.0,4.0,1.0,39.0,1,NOT_EVALUATED,DRAFT_PENDING_COORDINATOR
2,053614f6-2096-51a4-b238-38e8153a9ff8,Curitiba,Politécnica,2248,CECP 2021/2,2021-2,9d7bb182-8801-5f4b-a9bb-f06356a22ec1,7,Manhã,PCP100A,Arquitetura e Organização de Computadores,39.0,3.0,1.0,43.0,2,NOT_EVALUATED,DRAFT_PENDING_COORDINATOR
3,05aa7940-20a6-52bb-95fc-275de9ceda38,Curitiba,Politécnica,2248,CECP 2022/1,2022-1,b0841de6-0e13-5202-a0ed-25faca4cfba6,6,Manhã,PSI107A,Banco de Dados,40.0,3.0,2.0,45.0,1,NOT_EVALUATED,DRAFT_PENDING_COORDINATOR
4,08d478e7-78f5-5e49-95a6-5a5fa1678b58,Curitiba,Politécnica,2248,CECP 2021/2,2021-2,9d7bb182-8801-5f4b-a9bb-f06356a22ec1,7,Manhã,PCP116A,Sistemas de Computação,39.0,5.0,0.0,44.0,2,NOT_EVALUATED,DRAFT_PENDING_COORDINATOR
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
65,efa1e090-39f1-5b4b-9452-7e44a36c2034,Curitiba,Politécnica,2348,CECP 2020/1-N,2020-1,09f745e8-22f3-5311-ba63-2c22ed9088ae,10,Noite,P110A,Implementação e Operação de Projeto Transformador,34.0,4.0,1.0,39.0,15,NOT_EVALUATED,DRAFT_PENDING_COORDINATOR
66,f64a7930-4389-50f9-9625-45570c7c0bda,Curitiba,Politécnica,2287,CCBS 2024/1,2024-1,05e82e16-5175-5f58-9180-24898cababea,2,Noite,P129A,Criptografia Aplicada,49.0,5.0,1.0,55.0,1,NOT_EVALUATED,DRAFT_PENDING_COORDINATOR
67,f7c0ca73-773a-58b8-a6a1-5b5eee53404e,Curitiba,Politécnica,2248,CECP 2021/2,2021-2,9d7bb182-8801-5f4b-a9bb-f06356a22ec1,7,Manhã,PCP110A,Modelagem de Sistemas Computacionais,39.0,5.0,0.0,44.0,2,NOT_EVALUATED,DRAFT_PENDING_COORDINATOR
68,fcafb740-71bf-54d4-9ffa-abf49b756f28,Curitiba,Politécnica,2287,CCBS 2021/1,None,None,8,Noite,PCS107A,Experiência Criativa: Criando Soluções com Cib...,NaN,NaN,NaN,NaN,2,NOT_EVALUATED,QUARANTINED


**Executar:** filtrar ciclo/escola/curso/matriz/coorte/período/turno/disciplina/tipo/status e abrir uma oferta. **Resultado:** cada filtro faz nova consulta HTTP; detalhe contém atividades, demanda e lineage.

In [2]:

controls=explorer(proposal_id,payload)


**Evidência:** conferir P164 exclusivamente pela API. **Resultado:** uma oferta, três atividades; 41 planejados não são 82 estudantes.

In [3]:

circuitos=fetch(proposal_id,{'course_code':'2248','discipline_code':'P164'})
assert len(circuitos['offers'])==1
offer=circuitos['offers'][0]
assert sorted(a['reference_planned_students'] for a in offer['activities'])==[20,21,41]
display(table(circuitos['offers']))
display(pd.DataFrame(offer['activities']))


,offer_id,Estabelecimento,Escola,Curso,Grade,Ingresso,Coorte,Período,Turno,Disciplina,Descrição,Regular,Dependência,Eletiva,Total_previsto,Atividades,Unificação,Status
0,64050b10-e0c5-55a0-a771-ef84ab0a0fe8,Curitiba,Politécnica,2248,CECP 2023/1,2023-1,ea721911-589d-5f85-8f92-557670f6b085,4,Manhã,P164,Concepção de Circuitos Elétricos,43,4,1,48,3,NOT_EVALUATED,DRAFT_PENDING_COORDINATOR


,source_row_id,source_file_sha256,source_sheet,source_row_number,kind,division,reference_planned_students,reference_seats,capacity,capacity_status,proposed_groups,proposal_status
0,cdeb7107-d787-53f2-bdaf-3aced688e936,8c472d8ecf2468197a3cb980bf6d1e18f1297d43b651e2...,DISCIPLINAS REGULARES,10,THEORY,Completa,41,45,None,OFFICIAL_RULE_NOT_AVAILABLE,[],PENDING_OFFICIAL_CAPACITY
1,5379255c-189d-513f-86b1-ab4c1eae2a01,8c472d8ecf2468197a3cb980bf6d1e18f1297d43b651e2...,DISCIPLINAS REGULARES,11,PRACTICE,P2-1,21,22,None,OFFICIAL_RULE_NOT_AVAILABLE,[],PENDING_OFFICIAL_CAPACITY
2,56972ea9-1dd2-5623-a818-4594fc6d5c52,8c472d8ecf2468197a3cb980bf6d1e18f1297d43b651e2...,DISCIPLINAS REGULARES,12,PRACTICE,P2-2,20,23,None,OFFICIAL_RULE_NOT_AVAILABLE,[],PENDING_OFFICIAL_CAPACITY


**Onde conferir:** [MLflow](http://localhost:5000), [RustFS](http://localhost:9001), [API](http://localhost:8000/docs), [Logs](http://localhost:8088). Em falha: `docker compose logs --tail 100 api inference worker mlflow`.